# 모든 변화가 뉴스는 아니다 — 개표 상황 알림 (예시 데이터)
기준(역전, 격차 1%p 미만)을 바꿔 보며 알림이 언제 울리는지 확인해요.

In [ ]:
# ▶ 환경 준비 — Colab · VS Code 공통, 맨 처음 한 번 실행
import os, sys, platform, subprocess
IN_COLAB = 'google.colab' in sys.modules

def gpu_info():
    try:
        r = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                           capture_output=True, text=True, timeout=10)
        return r.stdout.strip() or None
    except Exception:
        return None

GPU = gpu_info()
print('실행 위치:', 'Colab' if IN_COLAB else f'내 컴퓨터 ({platform.system()})')
print('GPU:', GPU or '없음 → CPU로 실행 (이 노트북은 CPU로도 충분해요)')

# (선택) pandas를 GPU로 — Colab GPU 런타임 또는 Linux/WSL에 cuDF가 있을 때만
USE_GPU_PANDAS = False          # True로 바꾸고 이 칸을 다시 실행
if USE_GPU_PANDAS and GPU:
    try:
        get_ipython().run_line_magic('load_ext', 'cudf.pandas')
        print('pandas → GPU(cuDF) 사용')
    except Exception as e:
        print('cuDF를 쓸 수 없어 일반 pandas로 실행:', type(e).__name__)

# 한글 글꼴: Colab은 나눔고딕 설치, 윈도우는 맑은 고딕, 맥은 애플고딕
import matplotlib.pyplot as plt
from matplotlib import font_manager as fm
if IN_COLAB:
    nanum = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'
    if not os.path.exists(nanum):
        subprocess.run('apt-get -qq install -y fonts-nanum > /dev/null', shell=True)
    fm.fontManager.addfont(nanum)
    plt.rcParams['font.family'] = 'NanumGothic'
elif platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':
    plt.rcParams['font.family'] = 'AppleGothic'
else:                                   # 리눅스·WSL: 설치된 한글 글꼴 찾기
    ko = [f.name for f in fm.fontManager.ttflist if 'Nanum' in f.name or 'CJK KR' in f.name]
    if ko: plt.rcParams['font.family'] = ko[0]
plt.rcParams['axes.unicode_minus'] = False
print('한글 글꼴:', plt.rcParams['font.family'])

In [ ]:
import pandas as pd
stream = pd.DataFrame({
    '시각':   ['20:30', '21:00', '21:30', '22:00', '22:30', '23:00'],
    '개표율': [5, 15, 30, 50, 70, 90],
    '후보A':  [52.1, 50.8, 49.6, 48.9, 48.2, 47.9],
    '후보B':  [45.0, 46.9, 48.1, 49.3, 50.1, 50.6],
})
stream

In [ ]:
GAP = 1.0    # 초접전 기준(%p) — 2.0으로 바꿔 보기
prev = None
for _, r in stream.iterrows():
    gap = r['후보A'] - r['후보B']
    lead = 'A' if gap > 0 else 'B'
    alert = []
    if prev and lead != prev:
        alert.append(f'🔁 역전! 후보{lead} 선두')
    if abs(gap) < GAP:
        alert.append(f'⚠️ 격차 {abs(gap):.1f}%p 초접전')
    print(r['시각'], f"개표 {r['개표율']}%", ' / '.join(alert) or '-')
    prev = lead